# KOPIS 상세 재수집과 CLI

프로젝트 최상위의 **`KOPIS_수집기.cmd`를 더블클릭**하면 제어 메뉴가 열린다. 메뉴에서 1번을 선택하거나 `kopis.cmd run`으로 수집을 실행하고, 실행 창에서 **Q 또는 Ctrl+C**로 중지한다. 다른 터미널에서는 `kopis.cmd stop`으로 중지하고 `status`·`watch`로 현황을 확인한다. `run --limit 1`은 미수집 ID 한 건만 요청한다.

기본 호출 간격은 **1초**이며, 직전 응답이 끝난 뒤 기다리므로 이 프로세스는 초당 최대 1회 이하로 순차 호출한다. 사용자 지정에 따라 HTTP 400은 같은 ID를 3초 뒤에 재시도한다. 400이 계속되면 성공 또는 사용자 중지까지 반복하며, 그 외 접근 차단 응답에서는 전체 수집을 중단한다. 중지·재개 시 정상 저장된 ID는 다시 요청하지 않는다. 기존 원본과 이전 분석 결과는 보존한다.

이 노트북의 기본 실행은 오프라인 점검이다. `kopis-runtime` 태그 셀은 CLI에서도 그대로 사용하므로 별도의 `.py` 수집 소스를 만들지 않는다. 인증키는 프로젝트 외부 설정 또는 환경변수로만 읽는다.

이 노트북은 필요할 때만 사용하는 **수집 관리 도구**이며 최종 분석의 전체 실행에는 포함하지 않는다. 수집 및 스냅샷 등록은 공용 원자료와 `metadata/`에 기록된다. `final/metadata/`에 고정한 분석 입력은 자동 교체하지 않는다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import argparse, csv, hashlib, json, os, signal, sys, time, uuid
import urllib.request, urllib.parse, urllib.error
import xml.etree.ElementTree as ET
from contextlib import contextmanager

ROOT = Path(globals().get("KOPIS_PROJECT_ROOT", Path.cwd())).resolve()
if not (ROOT / "AGENTS.md").exists():
    ROOT = next(p for p in ROOT.parents if (p / "AGENTS.md").exists())
LEGACY = ROOT / "data/processed/kopis"
RAW = ROOT / "data/raw/kopis"
ADDITIONS = ROOT / "data/raw/kopis_recollection"
OUT = ROOT / "data/analysis/kopis_collection"
REPORTS = ROOT / "reports/kopis_collection"
BASE = "https://kopis.or.kr/openApi/restful/"
INTERVAL_SECONDS = 1.0
HTTP400_RETRY_SECONDS = 3.0
KST = timezone(timedelta(hours=9))
SPECS = {
    "facility": ("prfplc", "facility_details", "mt10id", "facilities_list_seoul.csv"),
    "performance": ("pblprfr", "performance_details", "mt20id", "performance_detail_candidates.csv"),
}

def now_kst():
    return datetime.now(KST).isoformat()

def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{os.getpid()}.tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

## 1. 원문과 재개 ID

XML의 공식 ID가 파일 ID와 일치할 때만 완료로 인정한다. 공연 후보의 시설명 연결은 수집 대상 선정용이며 공식 ID 연결과 다르다. 신규 원문은 회차별 폴더에 추가하고 기존 원문은 수정하지 않는다.

In [ ]:
def parse_detail(blob, idkey, requested_id):
    root = ET.fromstring(blob)
    rows = root.findall("db")
    if root.tag != "dbs" or len(rows) != 1:
        raise ValueError("unexpected_xml_structure")
    row = {e.tag: (e.text or "").strip() for e in rows[0] if len(e) == 0}
    if row.get(idkey) != requested_id:
        raise ValueError("detail_id_mismatch")
    return row

def load_inventory():
    targets, records, districts = {}, {}, {}
    for kind, (_, folder, idkey, filename) in SPECS.items():
        with (LEGACY / filename).open(encoding="utf-8-sig", newline="") as stream:
            rows = list(csv.DictReader(stream))
        targets[kind] = {r[idkey] for r in rows}
        if kind == "facility":
            districts = {r[idkey]: r.get("gugunnm", "") for r in rows}
        records[kind] = {}
        paths = list((RAW / folder).glob("*.xml")) + sorted(ADDITIONS.glob(f"*/{folder}/*.xml"))
        for path in paths:
            records[kind][path.stem] = parse_detail(path.read_bytes(), idkey, path.stem)
    return targets, records, districts

def update_counts(state, targets, records):
    state["final_counts"] = {k: len(targets[k] & records[k].keys()) for k in SPECS}
    state["pending_counts"] = {k: len(targets[k] - records[k].keys()) for k in SPECS}
    state["target_counts"] = {k: len(v) for k, v in targets.items()}
    state["updated_at_kst"] = now_kst()

def save_state(state):
    atomic_json(REPORTS / f"{state['run_id']}.json", state)
    atomic_json(REPORTS / "latest.json", state)

In [ ]:
def atomic_csv(path, rows, fields):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)

def export_tables(targets, records, districts):
    for kind, (_, _, idkey, _) in SPECS.items():
        rows = [dict(records[kind][i]) for i in sorted(records[kind])]
        if kind == "facility":
            for row in rows:
                row["gugunnm"] = districts.get(row[idkey], "")
        fields = list(dict.fromkeys(k for row in rows for k in row)) or [idkey]
        atomic_csv(OUT / f"{kind}_details.csv", rows, fields)
        pending = [{idkey: i} for i in sorted(targets[kind] - records[kind].keys())]
        atomic_csv(OUT / f"pending_{kind}_details.csv", pending, [idkey])

## 2. 실행 잠금과 안전한 중지

OS 파일 잠금으로 이 CLI·노트북의 동시 수집을 막는다. 프로세스가 종료되면 잠금도 해제된다. Q·Ctrl+C·다른 터미널의 `stop` 명령은 중지 요청을 남긴다. 진행 중인 HTTP 요청은 응답 또는 제한시간(30초)까지 기다린 뒤 저장·종료한다. 새 `run` 명령은 이전 중지 표시를 지우고 재개한다.

In [ ]:
@contextmanager
def collector_lock():
    REPORTS.mkdir(parents=True, exist_ok=True)
    path = REPORTS / "collector.lock"
    stream = path.open("a+b")
    acquired = False
    try:
        if path.stat().st_size == 0:
            stream.write(b"0")
            stream.flush()
        stream.seek(0)
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            acquired = True
        except OSError:
            pass
        yield acquired
    finally:
        if acquired:
            stream.seek(0)
            if os.name == "nt":
                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)
        stream.close()

def collector_active():
    if not (REPORTS / "collector.lock").exists():
        return False
    with collector_lock() as acquired:
        return not acquired

def request_stop():
    if not collector_active():
        print("실행 중인 수집기가 없습니다.")
        return 0
    atomic_json(REPORTS / "STOP", {"requested_at_kst": now_kst()})
    print("중지를 요청했습니다. 진행 중인 응답을 저장한 뒤 종료합니다(요청 제한시간 30초).")
    return 0

In [ ]:
def stop_requested(control):
    if control["stop"] or (REPORTS / "STOP").exists():
        return True
    if os.name == "nt" and sys.stdin.isatty():
        import msvcrt
        while msvcrt.kbhit():
            if msvcrt.getwch().lower() == "q":
                control["stop"] = True
                return True
    return False

def wait_between_requests(control, last_response_at, interval=None):
    interval = INTERVAL_SECONDS if interval is None else interval
    while last_response_at is not None:
        if stop_requested(control):
            return False
        remaining = interval - (time.monotonic() - last_response_at)
        if remaining <= 0:
            break
        time.sleep(min(remaining, 0.1))
    return not stop_requested(control)

def load_key():
    key = os.environ.get("KOPIS_API_KEY", "").strip()
    keyfile = Path.home() / ".config/grad/kopis.env"
    if not key and keyfile.exists():
        for line in keyfile.read_text(encoding="utf-8-sig").splitlines():
            name, sep, value = line.strip().partition("=")
            if sep and name == "KOPIS_API_KEY":
                key = value.strip().strip(chr(34)).strip(chr(39))
                break
    return key

class CollectionStopped(Exception):
    def __init__(self, reason, http_status=None, blocked_marker=False):
        self.reason, self.http_status, self.blocked_marker = reason, http_status, blocked_marker
        super().__init__(reason)

## 3. 요청·차단 검사

직렬 요청만 사용하고 리디렉션은 따르지 않는다. HTTP 400은 응답 종료 후 3초 대기해 같은 ID를 재시도한다. HTTP 401/403/429 및 400 이외의 차단 응답은 즉시 중단한다. 원문 오류와 인증키가 포함된 URL은 출력·저장하지 않는다. `http_requests_sent`는 실제 요청 시도 수이다.

In [ ]:
class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

def fetch_detail_once(kind, record_id, key, state):
    endpoint, folder, idkey, _ = SPECS[kind]
    safe_url = BASE + endpoint + "/" + record_id
    request = urllib.request.Request(safe_url + "?" + urllib.parse.urlencode({"service": key}),
                                     headers={"User-Agent": "GraduationResearch/1.0"})
    state["http_requests_sent"] += 1
    try:
        with urllib.request.build_opener(NoRedirect).open(request, timeout=30) as response:
            blob, status = response.read(), response.status
    except urllib.error.HTTPError as exc:
        blob, status = exc.read(), exc.code
    except Exception:
        raise CollectionStopped("network_error") from None
    blocked = any(s in blob.lower() for s in (b"request blocked", b"access denied",
                  b"access blocked", b"too many requests", b"rate limit", "접근 차단".encode()))
    if blocked or status in (400, 401, 403, 429):
        raise CollectionStopped("api_access_rejected", status, blocked)
    if status != 200:
        raise CollectionStopped("http_error", status)
    if key.encode() in blob:
        raise CollectionStopped("response_contains_credential", status)
    try:
        row = parse_detail(blob, idkey, record_id)
    except (ValueError, ET.ParseError):
        raise CollectionStopped("unexpected_xml_or_id", status) from None
    path = ADDITIONS / state["run_id"] / folder / f"{record_id}.xml"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream:
        stream.write(blob)
    metadata = {"endpoint": safe_url, "collected_at_kst": now_kst(), "http_status": status,
                "sha256": hashlib.sha256(blob).hexdigest(), "bytes": len(blob),
                "path": path.relative_to(ROOT).as_posix()}
    with path.with_suffix(".request.json").open("x", encoding="utf-8") as stream:
        json.dump(metadata, stream, ensure_ascii=False, indent=2)
    return row

HTTP 400 재시도는 같은 ID를 유지한다. 실제 HTTP 요청 수와 재시도 횟수를 따로 기록하고, 3초 대기는 0.1초 단위로 중지 요청을 확인한다. 400 응답을 수집 완료로 세거나 원문으로 저장하지 않는다.

In [ ]:
def fetch_detail(kind, record_id, key, state, control, printer):
    while True:
        if stop_requested(control):
            raise CollectionStopped("stopped_by_user")
        try:
            row = fetch_detail_once(kind, record_id, key, state)
            state.update(status="running", http_status=200, blocked_marker=False,
                         retrying_id=None, next_retry_at_kst=None)
            return row
        except CollectionStopped as exc:
            if exc.http_status != 400:
                raise
            retry_started = time.monotonic()
            state["http400_responses"] += 1
            state.update(status="retry_wait", http_status=400, blocked_marker=exc.blocked_marker,
                         retrying_id=record_id, estimated_remaining_seconds=None,
                         updated_at_kst=now_kst(),
                         next_retry_at_kst=(datetime.now(KST) + timedelta(seconds=HTTP400_RETRY_SECONDS)).isoformat())
            save_state(state)
            printer.show(state, force=True)
            if not wait_between_requests(control, retry_started, HTTP400_RETRY_SECONDS):
                raise CollectionStopped("stopped_by_user", 400, exc.blocked_marker) from None
            state["http400_retry_count"] += 1
            state.update(status="running", next_retry_at_kst=None, updated_at_kst=now_kst())
            save_state(state)


## 4. 진행률·남은 시간

전체 진행률의 분모는 서울 시설 1,755개와 기존 연구 후보 공연 13,727개이다. 서울 전체 공연 목록 30,047개와 구분한다. 예상 시간은 이번 실행의 실제 속도로 계산하며 5건 전까지는 계산 중으로 표시한다. 중단된 실행에는 완료 예정 시각을 표시하지 않는다. 1초는 응답 처리시간을 제외한 최소 대기이므로 실제 완료까지는 대기시간 합계보다 오래 걸린다.

In [ ]:
STATUS_LABELS = {
    "retry_wait": "HTTP 400 — 3초 뒤 재시도",
    "running": "수집 중", "completed_pending_queue": "수집 완료",
    "api_access_rejected": "서버 차단으로 중단", "stopped_by_user": "사용자가 중지",
    "interrupted": "사용자가 중지", "request_limit_reached": "지정 건수 수집 완료",
    "missing_credential": "인증키 없음", "http_error": "HTTP 오류로 중단",
    "network_error": "통신 오류로 중단", "local_error": "로컬 오류로 중단",
    "unexpected_xml_or_id": "응답 형식 오류로 중단",
}

def duration(seconds):
    if seconds is None:
        return "계산 중"
    minutes = int(max(0, seconds) / 60 + 0.999)
    return f"{minutes // 60}시간 {minutes % 60:02d}분"

def progress_line(state, active):
    done, pending = state["final_counts"], state["pending_counts"]
    total = sum(done.values()) + sum(pending.values())
    percent = 100 * sum(done.values()) / total if total else 100
    eta = duration(state.get("estimated_remaining_seconds")) if active else "중단/완료 상태"
    if active and state.get("status") == "retry_wait":
        eta = "HTTP 400: 3초 뒤 재시도"
    return (f"{percent:5.1f}% | 시설 {done['facility']:,}/{done['facility']+pending['facility']:,}"
            f" | 공연 {done['performance']:,}/{done['performance']+pending['performance']:,}"
            f" | 남음 {sum(pending.values()):,}건 | 예상 {eta}")

def print_status(state, active):
    label = STATUS_LABELS.get(state.get("status"), state.get("status", "기록 없음"))
    if state.get("status") in {"running", "retry_wait"} and not active:
        label = "프로세스 종료 감지 — 재개 시 원문에서 다시 계산"
    print(f"상태: {label} | 현재 실행: {'있음' if active else '없음'}")
    print(progress_line(state, active))
    print(f"다음 실행 간격: {INTERVAL_SECONDS}초 | 마지막 실행 간격: {state.get('request_interval_seconds')}초")
    print(f"HTTP 400 응답수: {state.get('http400_responses', '미기록')} | 재시도수: {state.get('http400_retry_count', '미기록')} | 재시도 대기: {HTTP400_RETRY_SECONDS:g}초")
    print(f"최근 갱신: {state.get('updated_at_kst', state.get('finished_at_kst', '-'))}")
    if state.get("http_status"):
        print(f"마지막 HTTP 상태: {state['http_status']}")
    if not active and sum(state['pending_counts'].values()):
        print(f"재개 후 최소 대기 합계: {duration(sum(state['pending_counts'].values()) * INTERVAL_SECONDS)} + 응답·저장 시간")

class ProgressPrinter:
    def __init__(self):
        self.last_print = -float("inf")
    def show(self, state, force=False):
        if not force and time.monotonic() - self.last_print < 1:
            return
        self.last_print = time.monotonic()
        line = progress_line(state, state["status"] in {"running", "retry_wait"})
        if sys.stdout.isatty():
            print("\r\033[2K" + line, end="", flush=True)
        else:
            print(line, flush=True)
    def finish(self):
        if sys.stdout.isatty():
            print()

## 5. 순차 수집 루프

1초 간격은 모든 ID 요청에 같은 방식으로 적용한다. 응답·저장이 오래 걸리면 추가 호출을 몰아서 보내지 않는다. 진행 JSON은 매 건, 통합 CSV와 미수집 목록은 25건마다 및 종료 시 갱신한다. HTTP 400 재시도 대기 중에는 예상 시간 대신 재시도 상태를 표시한다. 중지 상태에서는 남은 시간이 자동으로 계속 줄어들지 않는다.

In [ ]:
def initial_state(targets, records, limit):
    stamp = datetime.now(KST).strftime("%Y%m%d_%H%M%S")
    state = {"run_id": stamp + "_" + uuid.uuid4().hex[:6],
             "started_at_kst": now_kst(), "network_enabled": True, "pid": os.getpid(),
             "status": "running", "requests_attempted": 0, "http_requests_sent": 0,
             "new_facility_details": 0, "new_performance_details": 0,
             "http400_responses": 0, "http400_retry_count": 0,
             "http400_retry_seconds": HTTP400_RETRY_SECONDS,
             "request_interval_seconds": INTERVAL_SECONDS, "max_requests": limit or None}
    update_counts(state, targets, records)
    state["initial_counts"] = dict(state["final_counts"])
    return state

def update_timing(state, started):
    successes = state["new_facility_details"] + state["new_performance_details"]
    elapsed = time.monotonic() - started
    state["elapsed_seconds"] = round(elapsed, 2)
    state["seconds_per_record"] = elapsed / successes if successes >= 5 else None
    state["estimated_remaining_seconds"] = (
        round(state["seconds_per_record"] * sum(state["pending_counts"].values()))
        if state["seconds_per_record"] is not None and state["status"] == "running" else None)

def collect(limit=0):
    with collector_lock() as acquired:
        if not acquired:
            print("이미 수집기가 실행 중입니다. status/watch로 확인하거나 stop으로 중지하세요.")
            return 2
        return collect_locked(limit)

In [ ]:
def collect_locked(limit):
    (REPORTS / "STOP").unlink(missing_ok=True)
    targets, records, districts = load_inventory()
    state = initial_state(targets, records, limit)
    control = {"stop": False}
    old_handler = signal.signal(signal.SIGINT, lambda *_: control.update(stop=True))
    printer = ProgressPrinter()
    started, last_response_at = time.monotonic(), None
    key = ""
    print(f"KOPIS 수집 시작 | 간격 {INTERVAL_SECONDS}초 | Q 또는 Ctrl+C로 중지", flush=True)
    try:
        key = load_key() if sum(state["pending_counts"].values()) else ""
        if not key and sum(state["pending_counts"].values()):
            raise CollectionStopped("missing_credential")
        save_state(state)
        printer.show(state, force=True)
        for kind in SPECS:
            pending = sorted(targets[kind] - records[kind].keys())
            for record_id in pending:
                if limit and state["requests_attempted"] >= limit:
                    raise CollectionStopped("request_limit_reached")
                if not wait_between_requests(control, last_response_at):
                    raise CollectionStopped("stopped_by_user")
                state["last_kind"], state["last_id"] = kind, record_id
                state["requests_attempted"] += 1
                records[kind][record_id] = fetch_detail(kind, record_id, key, state, control, printer)
                last_response_at = time.monotonic()
                state[f"new_{kind}_details"] += 1
                update_counts(state, targets, records)
                update_timing(state, started)
                save_state(state)
                if state["requests_attempted"] % 25 == 0:
                    export_tables(targets, records, districts)
                printer.show(state)
        state["status"] = "completed_pending_queue"
    except CollectionStopped as exc:
        state.update(status=exc.reason, http_status=exc.http_status,
                     blocked_marker=exc.blocked_marker, stopped_without_retry=True)
    except KeyboardInterrupt:
        state.update(status="stopped_by_user", stopped_without_retry=True)
    except Exception as exc:
        state.update(status="local_error", error_type=type(exc).__name__, stopped_without_retry=True)
    finally:
        key = ""
        state["next_retry_at_kst"] = None
        state["retrying_id"] = None
        state["finished_at_kst"] = now_kst()
        update_counts(state, targets, records)
        update_timing(state, started)
        try:
            save_state(state)
            export_tables(targets, records, districts)
        finally:
            signal.signal(signal.SIGINT, old_handler)
            printer.finish()
    print_status(state, active=False)
    return 0 if state["status"] in {"completed_pending_queue", "stopped_by_user", "request_limit_reached"} else 3

## 6. CLI 명령

프로젝트 최상위 `KOPIS_수집기.cmd`를 더블클릭하면 메뉴가 열리고, 수집이 중지되거나 완료되면 메뉴로 돌아온다. 0번은 창을 닫고 5번은 결과 폴더를 연다. 더블클릭만으로 API를 호출하지는 않는다. 명령어 방식은 프로젝트 폴더의 PowerShell 또는 명령 프롬프트에서 실행한다. `run`/`start`는 현재 창에서 실행하므로 진행률이 바로 보인다. 창이 닫히면 중단될 수 있으므로 정상 종료에는 Q 또는 `stop`을 사용한다. OS가 강제 종료하더라도 다음 실행은 남아 있는 정상 XML에서 재개한다.

```powershell
.\kopis.cmd run             # 실행·재개: 1초 간격
.\kopis.cmd run --limit 1   # 한 건만 확인
.\kopis.cmd status          # 현재 상태·남은 건수
.\kopis.cmd watch           # 별도 창에서 실시간 확인
.\kopis.cmd stop            # 실행 중인 수집기에 중지 요청
```

`watch` 창의 Ctrl+C는 현황 보기만 종료하며 수집 자체는 유지한다.

In [ ]:
def read_status():
    path = REPORTS / "latest.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    targets, records, _ = load_inventory()
    state = {"status": "offline_inventory", "request_interval_seconds": INTERVAL_SECONDS}
    update_counts(state, targets, records)
    return state

def watch_status():
    print("실시간 현황 보기 | Ctrl+C는 보기만 종료합니다.")
    printer = ProgressPrinter()
    try:
        while True:
            state, active = read_status(), collector_active()
            if not active:
                printer.finish()
                print_status(state, active=False)
                return 0
            printer.show(state, force=True)
            time.sleep(1)
    except KeyboardInterrupt:
        printer.finish()
        print("현황 보기를 종료했습니다. 수집을 멈추려면 stop을 실행하세요.")
        return 0

def interactive_menu():
    print(f"\nKOPIS 수집기 | 호출 간격 {INTERVAL_SECONDS}초 | HTTP 400: {HTTP400_RETRY_SECONDS:g}초 후 재시도")
    print(f"프로젝트: {ROOT}")
    print("실행 중에는 Q 또는 Ctrl+C로 중지하고 이 메뉴로 돌아옵니다.")
    while True:
        print("\n" + "─" * 58)
        print_status(read_status(), collector_active())
        print("\n  1  수집 실행 / 재개")
        print("  2  현재 현황 새로고침")
        print("  3  실시간 현황 보기")
        print("  4  실행 중인 수집 중지")
        print("  5  수집 결과 폴더 열기")
        print("  0  창 닫기")
        try:
            choice = input("\n번호 선택 > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n수집기 메뉴를 닫습니다.")
            return 0
        if choice == "0":
            return 0
        if choice == "2":
            continue
        if choice == "5":
            if OUT.exists() and os.name == "nt":
                os.startfile(str(OUT))
            else:
                print(f"결과 위치: {OUT}")
            continue
        action = {"1": "run", "3": "watch", "4": "stop"}.get(choice)
        if action:
            kopis_cli_main([action])
        else:
            print("0~5 중에서 선택하세요.")

def kopis_cli_main(argv=None):
    parser = argparse.ArgumentParser(prog="kopis.cmd", description="KOPIS 상세 수집: 1초 간격, HTTP 400은 3초 후 재시도")
    parser.add_argument("command", nargs="?", default="menu", choices=["menu", "run", "start", "status", "watch", "stop"])
    parser.add_argument("--limit", type=int, default=0, help="run/start에서 최대 요청할 ID 수(0=전체)")
    args = parser.parse_args(argv)
    if args.limit < 0 or (args.limit and args.command not in {"run", "start"}):
        parser.error("--limit은 run/start에서만 0 이상의 정수로 지정합니다.")
    try:
        if args.command == "menu":
            return interactive_menu()
        if args.command in {"run", "start"}:
            return collect(args.limit)
        if args.command == "stop":
            return request_stop()
        if args.command == "watch":
            return watch_status()
        print_status(read_status(), collector_active())
        return 0
    except KeyboardInterrupt:
        print("작업을 중단했습니다.")
        return 130
    except Exception as exc:
        print(f"실행 오류: {type(exc).__name__}. 프로젝트 경로와 파일 접근 권한을 확인하세요.")
        return 1

## 7. 현재 확보 현황 (오프라인)

아래 표는 저장된 원문에서 다시 계산한다. CLI는 실행 중 매 건 갱신되는 JSON을 읽는다. 이전 실행의 간격은 기록 그대로 보존하고 새 실행에는 1초를 적용한다.

In [ ]:
import pandas as pd
from IPython.display import display

inventory_targets, inventory_records, _ = load_inventory()
summary = pd.DataFrame([{
    "자료": "시설 상세" if k == "facility" else "연구 후보 공연 상세",
    "대상": len(inventory_targets[k]),
    "수집 완료": len(inventory_targets[k] & inventory_records[k].keys()),
    "미수집": len(inventory_targets[k] - inventory_records[k].keys()),
} for k in SPECS])
display(summary)
print_status(read_status(), collector_active())

해석: 현재 자료는 일부 수집 상태일 수 있다. 미수집 좌표·시설 이전 가능성과 현재 경계의 한계는 남는다. 시설명 임시 연결과 상세 XML의 공식 ID 연결을 구분하며, 기존 분석·보고서는 자동 갱신하지 않는다. 월별 공연 통계와 분기 매출을 일별 자료처럼 해석하지 않는다.

`Request Blocked`라는 응답만으로 초당 요청 제한이 원인이라고 단정할 수 없다. 1초 간격·단일 프로세스 잠금은 이 수집기를 제어하며, 같은 공인 IP를 쓰는 다른 프로그램의 호출까지 제어하지는 못한다.

분석에는 수집 중인 CSV를 직접 사용하지 않는다. 종료된 수집본을 `03_KOPIS_스냅샷검증.ipynb`에서 검증하고 등록한 고정 스냅샷·입력 매니페스트를 사용한다. 기존 두 업종 결과의 계산 검증에는 `data/analysis/baseline_two_industry/`를 사용한다.